# Apartment Features Debugger
Run the cells below to load the raw CSV and the generated features `.pkl` file. You can query any apartment ID to see how its descriptive text was mapped into the 134-dimensional feature space.

In [1]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Adjust the path if you run this notebook from a different directory
csv_path = "robinreal_datathon_challenge_2026/raw_data/structured_data_withoutimages-1776412361239.csv"
pkl_path = "robinreal_datathon_challenge_2026/raw_data/apartment_features_134.pkl"

df_raw = pd.read_csv(csv_path, encoding="utf-8-sig")
df_features = pd.read_pickle(pkl_path)

print(f"Loaded {len(df_raw)} raw rows and {len(df_features)} feature rows.")

Loaded 6757 raw rows and 6757 feature rows.


In [2]:
def virtualize_apartment(apartment_id):
    if apartment_id not in df_raw["id"].values:
        print(f"Apartment ID {apartment_id} not found.")
        return

    raw_row = df_raw[df_raw["id"] == apartment_id].iloc[0]
    feature_row = df_features[df_features["id"] == apartment_id].iloc[0]

    print("="*60)
    print(f"🏡 RAW DATA (ID: {apartment_id})")
    print("="*60)
    for col in ["title", "object_type_text", "price", "number_of_rooms", "area", "floor", "year_built", "object_street", "object_city", "distance_public_transport", "distance_shop"]:
        if col in raw_row:
            print(f"{col.ljust(25)}: {raw_row[col]}")

    print("\n--- DESCRIPTION ---")
    desc = str(raw_row.get("object_description", "No description."))
    print(desc[:1500] + ("...[TRUNCATED]" if len(desc) > 1500 else ""))

    print("\n" + "="*60)
    print("📊 GENERATED 134 FEATURES")
    print("="*60)

    features_series = feature_row.drop("id")

    print("--- NUMERICAL & CATEGORICAL FEATURES ---")
    for col in ["price", "area", "number_of_rooms", "floor", "year_built", "distance_public_transport", "distance_shop", "geo_lat", "geo_lng", "maybe_temporary", "offer_type"]:
        if col in features_series.index:
            print(f"{col.ljust(30)}: {features_series[col]}")

    print("\n--- BOOLEAN FEATURES ---")
    true_feats = features_series[features_series == True].index.tolist()
    false_feats = features_series[features_series == False].index.tolist()
    na_feats = features_series[pd.isna(features_series)].index.tolist()

    print(f"✅ TRUE FEATURES ({len(true_feats)}):")
    print(", ".join(true_feats))
    print(f"\n❌ FALSE FEATURES ({len(false_feats)}):")
    print(", ".join(false_feats))
    print(f"\n⚪ UNDEFINED/NA FEATURES ({len(na_feats)}):")
    for i in range(0, len(na_feats), 5):
        print(", ".join(na_feats[i:i+5]))


In [5]:
# Example usage with a random ID from the dataset
sample_id = df_raw["id"].sample(1).iloc[0]
virtualize_apartment(sample_id)

🏡 RAW DATA (ID: 3795)
title                    : Depot / Geschäftsräume
object_type_text         : Gewerbeobjekt
price                    : 195
number_of_rooms          : nan
area                     : nicht verfügbar
floor                    : nan
year_built               : nan
object_street            : nan
object_city              : nan
distance_public_transport: nan
distance_shop            : nan

--- DESCRIPTION ---
Keller/Lager im Keller mit Stromanschluss.<br /><br />Dank seiner privilegierten Lage profitieren Sie von einem einfachen Zugang zu Geschäften, Restaurants und öffentlichen Verkehrsmitteln, während Sie in einem ruhigen und funktionalen Raum wohnen.

📊 GENERATED 134 FEATURES
--- NUMERICAL & CATEGORICAL FEATURES ---
price                         : 195
area                          : nan
number_of_rooms               : nan
floor                         : nan
year_built                    : nan
distance_public_transport     : nan
distance_shop                 : nan
geo_lat

In [ ]:
# Interactive Visualizer Widget (requires ipywidgets)
id_input = widgets.Text(description="Apartment ID:", value=str(sample_id))
button = widgets.Button(description="Visualize")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        virtualize_apartment(id_input.value)

button.on_click(on_click)
display(id_input, button, output)